# 09 — Estatísticas descritivas do FINDRISC

## 1. Objetivo

Calcular média, moda, mediana, desvio-padrão amostral, P25 e P75 do FINDRISC somente entre pacientes com pelo menos uma doença autoimune explicitamente registrada. Valores ausentes são preservados e não entram nos cálculos.

## 2. Importações

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

## 3. Configuração

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "pacientes_clean.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.statistics import descriptive_numeric
from src.variables import count_autoimmune_diagnoses

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "pacientes_clean.csv"

## 4. Carregamento

Somente o escore FINDRISC e o diagnóstico padronizado são carregados do dataset anonimizado. Nenhum registro individual ou identificador é exibido.

In [3]:
df = pd.read_csv(
    PROCESSED_PATH,
    usecols=["diagnostico_padronizado", "findrisc_score"],
)
n_diagnosticos_autoimunes = count_autoimmune_diagnoses(df["diagnostico_padronizado"])
autoimmune_mask = n_diagnosticos_autoimunes.ge(1).fillna(False)
autoimmune = df.loc[autoimmune_mask].copy()
findrisc = pd.to_numeric(autoimmune["findrisc_score"], errors="coerce")

print(f"Pacientes no dataset: {len(df)}")
print(f"Pacientes com doença autoimune explícita: {len(autoimmune)}")
print(f"FINDRISC válido no grupo autoimune: {findrisc.notna().sum()}")
print(f"FINDRISC ausente no grupo autoimune: {findrisc.isna().sum()}")

Pacientes no dataset: 75
Pacientes com doença autoimune explícita: 69
FINDRISC válido no grupo autoimune: 68
FINDRISC ausente no grupo autoimune: 1


## 5. Validações

In [4]:
assert df.columns.tolist() == ["diagnostico_padronizado", "findrisc_score"]
assert len(df) == 75
assert len(autoimmune) == 69
assert n_diagnosticos_autoimunes.eq(1).sum() == 68
assert n_diagnosticos_autoimunes.gt(1).sum() == 1
assert n_diagnosticos_autoimunes.eq(0).sum() == 6
assert findrisc.notna().sum() == 68
assert findrisc.isna().sum() == 1
assert findrisc.dropna().between(0, 30).all()
print("Filtro autoimune e escores FINDRISC validados.")

Filtro autoimune e escores FINDRISC validados.


## 6. Análise

O desvio-padrão é amostral (`ddof=1`). As estatísticas usam apenas os 68 escores válidos entre os 69 pacientes com doença autoimune explícita; o valor ausente não é substituído nem convertido em zero.

In [5]:
summary = descriptive_numeric(findrisc)

result_table = pd.DataFrame(
    {
        "Medida": [
            "Pacientes com doença autoimune",
            "FINDRISC válido no grupo autoimune",
            "FINDRISC ausente no grupo autoimune",
            "Média",
            "Moda",
            "Mediana",
            "Desvio-padrão",
            "P25",
            "P75",
        ],
        "Resultado": [
            str(len(autoimmune)),
            str(summary["n_valido"]),
            str(summary["n_ausente"]),
            f'{summary["media"]:.2f} pontos'.replace(".", ","),
            f'{summary["moda"]} pontos',
            f'{summary["mediana"]:.2f} pontos'.replace(".", ","),
            f'{summary["desvio_padrao"]:.2f} pontos'.replace(".", ","),
            f'{summary["p25"]:.2f} pontos'.replace(".", ","),
            f'{summary["p75"]:.2f} pontos'.replace(".", ","),
        ],
    }
)

for measure, result in result_table.itertuples(index=False, name=None):
    print(f"{measure}: {result}")

Pacientes com doença autoimune: 69
FINDRISC válido no grupo autoimune: 68
FINDRISC ausente no grupo autoimune: 1
Média: 13,65 pontos
Moda: 9 pontos
Mediana: 13,50 pontos
Desvio-padrão: 6,19 pontos
P25: 9,00 pontos
P75: 19,00 pontos


## 7. Visualização

Tabela final das estatísticas solicitadas:

In [6]:
display(result_table.style.hide(axis="index"))

Medida,Resultado
Pacientes com doença autoimune,69
FINDRISC válido no grupo autoimune,68
FINDRISC ausente no grupo autoimune,1
Média,"13,65 pontos"
Moda,9 pontos
Mediana,"13,50 pontos"
Desvio-padrão,"6,19 pontos"
P25,"9,00 pontos"
P75,"19,00 pontos"


## 8. Conclusões deste notebook

In [7]:
assert summary["moda"] == "9"
assert summary["p25"] == 9.00
assert summary["p75"] == 19.00
print(
    "Entre os 69 pacientes com doença autoimune explícita, 68 possuíam "
    "FINDRISC válido. A média foi 13,65 pontos, a mediana foi 13,50 pontos "
    "e a metade central dos escores ficou entre 9,00 e 19,00 pontos."
)

Entre os 69 pacientes com doença autoimune explícita, 68 possuíam FINDRISC válido. A média foi 13,65 pontos, a mediana foi 13,50 pontos e a metade central dos escores ficou entre 9,00 e 19,00 pontos.


## 9. Outputs gerados

In [8]:
print("Tabela exibida neste notebook.")

Tabela exibida neste notebook.
